In [41]:
import pandas as pd
import os

data_path = "../data/raw/"
files = [f for f in os.listdir(data_path) if f.endswith(".csv")]

files

['Youtube01-Psy.csv',
 'Youtube02-KatyPerry.csv',
 'Youtube03-LMFAO.csv',
 'Youtube04-Eminem.csv',
 'Youtube05-Shakira.csv']

In [42]:
df_list = []

for file in files:
    file_path = os.path.join(data_path, file)
    temp_df = pd.read_csv(file_path)
    df_list.append(temp_df)

    df = pd.concat(df_list, ignore_index=True)

In [43]:
print("Shape:", df.shape)
print("Columns:", df.columns)
print(df.head())


Shape: (1956, 5)
Columns: Index(['COMMENT_ID', 'AUTHOR', 'DATE', 'CONTENT', 'CLASS'], dtype='object')
                                    COMMENT_ID            AUTHOR  \
0  LZQPQhLyRh80UYxNuaDWhIGQYNQ96IuCg-AYWqNPjpU         Julius NM   
1  LZQPQhLyRh_C2cTtd9MvFRJedxydaVW-2sNg5Diuo4A       adam riyati   
2  LZQPQhLyRh9MSZYnf8djyk0gEF9BHDPYrrK-qCczIY8  Evgeny Murashkin   
3          z13jhp0bxqncu512g22wvzkasxmvvzjaz04   ElNino Melendez   
4          z13fwbwp1oujthgqj04chlngpvzmtt3r3dw            GsMega   

                  DATE                                            CONTENT  \
0  2013-11-07T06:20:48  Huh, anyway check out this you[tube] channel: ...   
1  2013-11-07T12:37:15  Hey guys check out my new channel and our firs...   
2  2013-11-08T17:34:21             just for test I have to say murdev.com   
3  2013-11-09T08:28:43   me shaking my sexy ass on my channel enjoy ^_^ ﻿   
4  2013-11-10T16:05:38            watch?v=vtaRGgvGtWQ   Check this out .﻿   

   CLASS  
0      1  
1   

In [44]:
df["CLASS"].value_counts()

CLASS
1    1005
0     951
Name: count, dtype: int64

In [45]:
# Percentage of each class
df["CLASS"].value_counts(normalize=True)


CLASS
1    0.513804
0    0.486196
Name: proportion, dtype: float64

In [46]:
# show random spam comments
df[df["CLASS"] == 1]["CONTENT"].sample(5)

1054                    Check out this video on YouTube:﻿
1011                    Check out this video on YouTube:﻿
247     It's so hard, sad :( iThat little child Actor ...
1140    my sister just received over 6,500 new <a rel=...
1051                   Subscribe I&#39;ll subscribe back﻿
Name: CONTENT, dtype: object

In [47]:
# show random non spam comments
df[df["CLASS"] == 0]["CONTENT"].sample(5)

1336                                   2015 and more....﻿
1856               I always have goose bumps at that part
1673                                            GREAT!!!﻿
844                            cooooooooooooolllllllllll﻿
1328    Every collaboration between them, we know it w...
Name: CONTENT, dtype: object

In [48]:
import re

# function to clean text
def clean_text(text):

    # convert text to lowercase
    text = text.lower()

    # remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # remove punctuation and special characters
    text = re.sub(r"[^a-z\s]", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [49]:
df ["clean_comment"] = df["CONTENT"].apply(clean_text)
df.head()

,COMMENT_ID,AUTHOR,DATE,CONTENT,CLASS,clean_comment
0,LZQPQhLyRh80UYxNuaDWhIGQYNQ96IuCg-AYWqNPjpU,Julius NM,2013-11-07T06:20:48,"Huh, anyway check out this you[tube] channel: ...",1,huh anyway check out this youtube channel koby...
1,LZQPQhLyRh_C2cTtd9MvFRJedxydaVW-2sNg5Diuo4A,adam riyati,2013-11-07T12:37:15,Hey guys check out my new channel and our firs...,1,hey guys check out my new channel and our firs...
2,LZQPQhLyRh9MSZYnf8djyk0gEF9BHDPYrrK-qCczIY8,Evgeny Murashkin,2013-11-08T17:34:21,just for test I have to say murdev.com,1,just for test i have to say murdevcom
3,z13jhp0bxqncu512g22wvzkasxmvvzjaz04,ElNino Melendez,2013-11-09T08:28:43,me shaking my sexy ass on my channel enjoy ^_^ ﻿,1,me shaking my sexy ass on my channel enjoy
4,z13fwbwp1oujthgqj04chlngpvzmtt3r3dw,GsMega,2013-11-10T16:05:38,watch?v=vtaRGgvGtWQ Check this out .﻿,1,watchvvtarggvgtwq check this out


In [50]:
from sklearn.model_selection import train_test_split

# features & labels
x = df["clean_comment"]
y = df["CLASS"]

# split dataset
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
    )

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

# initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words="english")

# fit on training data
x_train_tfidf = vectorizer.fit_transform(x_train)

# transform test data
x_test_tfidf = vectorizer.transform(x_test)

In [52]:
x_train_tfidf.shape

(1564, 3105)

In [53]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(x_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [54]:
y_pred = model.predict (x_test_tfidf)

In [55]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8928571428571429


In [57]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [58]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[150  26]
 [ 16 200]]


In [59]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.85      0.88       176
           1       0.88      0.93      0.90       216

    accuracy                           0.89       392
   macro avg       0.89      0.89      0.89       392
weighted avg       0.89      0.89      0.89       392



In [60]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

log_probs = model.feature_log_prob_

top_spam_indices = np.argsort(log_probs[1])[-20:]

top_spam_words = feature_names[top_spam_indices]

print(top_spam_words)


['song' 'thank' 'make' 'money' 'music' 'hey' 'videos' 'new' 'comment'
 'just' 'im' 'guys' 'br' 'playlist' 'like' 'channel' 'subscribe' 'video'
 'youtube' 'check']


In [61]:
top_spam_words = feature_names[np.argsort(log_probs[1])[-20:]][::-1]
print(top_spam_words)

['check' 'youtube' 'video' 'subscribe' 'channel' 'like' 'playlist' 'br'
 'guys' 'im' 'just' 'comment' 'new' 'videos' 'hey' 'music' 'money' 'make'
 'thank' 'song']


In [63]:
def predict_spam(comment):

    cleaned = clean_text(comment)

    vectorized = vectorizer.transform([cleaned])

    prediction = model.predict(vectorized)[0]

    if prediction == 1:
        return "Spam"
    else:
        return "Not Spam"

In [64]:
predict_spam("subscribe to my channel")

'Spam'

In [65]:
predict_spam("this song is amazing")

'Not Spam'

In [ ]:
import pandas as pd

# create a dataframe to compare predictions
results = pd.DataFrame({
    "comment": x_test,
    "actual": y_test,
    "predicted": y_pred
})

In [67]:
# filter rows where prediction is wrong
errors = results[results["actual"] != results["predicted"]]

errors.head(10)


,comment,actual,predicted
374,i really love this video,1,0
785,br i like video,0,1
1807,i want new song,0,1
1718,help shakiras waka waka be the first song by a...,1,0
1531,i lovet,0,1
70,billions in,0,1
1383,dress like rihanna at kpopcitynet the largest ...,1,0
678,i hate videos like these with those poor animals,0,1
1399,first they were fighting then they were making...,0,1
782,why sooooo many downs,0,1


In [68]:
len(errors)

42

In [74]:
errors[errors["actual"] == 1].head(10)

,comment,actual,predicted
374,i really love this video,1,0
1718,help shakiras waka waka be the first song by a...,1,0
1383,dress like rihanna at kpopcitynet the largest ...,1,0
1355,check out fantasy music right here gt thefanta...,1,0
353,this is the song,1,0
128,discover a beautiful song of a young moroccan,1,0
544,this song is so addicting the hook is dope and...,1,0
943,we can have a party next share,1,0
555,i love katy fashions tiger care to visit my bl...,1,0
1226,finally someone shares the same opinion as me ...,1,0


In [75]:
from sklearn.linear_model import LogisticRegression

# Initialize model
log_model = LogisticRegression(max_iter=1000)

# Train model
log_model.fit(x_train_tfidf, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
y_pred_log = log_model.predict(x_test_tfidf)

In [77]:
from sklearn.metrics import accuracy_score

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

Logistic Regression Accuracy: 0.8877551020408163


In [81]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()

svm_model.fit(x_train_tfidf, y_train)

y_pred_svm = svm_model.predict(x_test_tfidf)

In [82]:
from sklearn.metrics import accuracy_score

svm_acc = accuracy_score(y_test, y_pred_svm)
print("SVM Accuracy:", svm_acc)

SVM Accuracy: 0.8928571428571429


In [ ]:
nb_acc = accuracy_score(y_test, y_pred)
log_acc = accuracy_score(y_test, y_pred_log)

comparison = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "Linear SVM"],
    "Accuracy": [nb_acc, log_acc, svm_acc]
})

comparison

,Model,Accuracy
0,Naive Bayes,0.892857
1,Logistic Regression,0.887755
2,Linear SVM,0.892857


In [84]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.83      0.95      0.89       176
           1       0.95      0.85      0.90       216

    accuracy                           0.89       392
   macro avg       0.89      0.90      0.89       392
weighted avg       0.90      0.89      0.89       392



 
 #### Cross Validation